# "Don't Pass@$k$"; Bayes@$N$

In this tutorial, we see five examples of `scorio.eval`:

1. Bayes@N in binary evaluation.
2. Greedy run as prior knowledge for `top-p` samples.
3. Categorical evaluation with `correctness` and `token_ratio`.
4. Uncertainty-aware model comparison with Bayes@$N$.
5. Intervals for Pass@$k$ and mG-Pass@$k$.

- Paper: https://arxiv.org/abs/2510.04265
- API docs: https://scorio.readthedocs.io/en/latest/api/eval.html

All result matrices use **rows = questions** and **columns = trials**. In notation, $R \in \{0, \ldots, C\}^{M \times N}$, where $M$ is the number of questions and $N$ is the number of trials per question.


## Scorio Eval API Quick Reference

Function names link to their specific API documentation entries, and source links point to the GitHub implementation files. The Bayes@$N$ rows correspond to the paper's main Bayesian framework; the pass-family interval helpers are Scorio API companions.

| Function (ReadThe Docs) | Source (GitHub) | Returns | Description |
| --- | --- | --- | --- |
| [`bayes(R, w, R0)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.bayes.bayes) | [`bayes.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/bayes.py) | $(\mu, \sigma)$ | Bayesian posterior mean and uncertainty |
| [`bayes_ci(R, w, R0)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.bayes.bayes_ci) | [`bayes.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/bayes.py) | $(\mu, \sigma, \mathrm{lo}, \mathrm{hi})$ | Bayesian posterior mean and uncertainty + credible interval |
| [`avg(R, w)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.avg.avg) | [`avg.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/avg.py) | $(a, \sigma_a)$ | Weighted average and uncertainty |
| [`avg_ci(R, w)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.avg.avg_ci) | [`avg.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/avg.py) | $(a, \sigma_a, \mathrm{lo}, \mathrm{hi})$ | Weighted average and uncertainty + credible interval |
| [`pass_at_k(R, k)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.pass_at_k.pass_at_k) | [`pass_at_k.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/pass_at_k.py) | $p$ | Pass@$k$ estimate |
| [`pass_at_k_ci(R, k)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.pass_at_k.pass_at_k_ci) | [`pass_at_k.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/pass_at_k.py) | $(\mu, \sigma, \mathrm{lo}, \mathrm{hi})$ | Pass@$k$ estimate + credible interval |
| [`pass_hat_k(R, k)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.pass_at_k.pass_hat_k) | [`pass_at_k.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/pass_at_k.py) | $p$ | Pass^$k$ |
| [`pass_hat_k_ci(R, k)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.pass_at_k.pass_hat_k_ci) | [`pass_at_k.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/pass_at_k.py) | $(\mu, \sigma, \mathrm{lo}, \mathrm{hi})$ | Pass^$k$ + credible interval |
| [`g_pass_at_k_tau(R, k, tau)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.gpass.g_pass_at_k_tau) | [`gpass.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/gpass.py) | $p$ | G-Pass@$k_{\tau}$ |
| [`mg_pass_at_k(R, k)`](https://scorio.readthedocs.io/en/latest/api/eval.html#scorio.eval.gpass.mg_pass_at_k) | [`gpass.py`](https://github.com/mohsenhariri/scorio/blob/main/scorio/eval/gpass.py) | $p$ | mG-Pass@$k$ |


In [ ]:
import math
import numpy as np
from scorio import eval

np.set_printoptions(suppress=True)


def print_ci(name, ci):
    mu, sigma, lo, hi = ci
    print(
        f"{name}: mu={mu:.4f}, sigma={sigma:.4f}, "
        f"95% CrI=[{lo:.4f}, {hi:.4f}], width={hi - lo:.4f}"
    )


def ranking_confidence(mu1, sigma1, mu2, sigma2):
    z = abs(mu1 - mu2) / math.sqrt(sigma1**2 + sigma2**2)
    rho = 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))
    return z, rho


def intervals_overlap(ci1, ci2):
    lo1, hi1 = ci1[2], ci1[3]
    lo2, hi2 = ci2[2], ci2[3]
    return not (hi1 < lo2 or hi2 < lo1)


def matrix_from_counts(counts, n_trials):
    return np.vstack([np.array([1] * c + [0] * (n_trials - c), dtype=int) for c in counts])


## Example 1: Bayes@N in Binary Evaluation

In this example, each trial is either correct (`1`) or wrong (`0`). A row is one question, and a column is one sampled answer for that question.

We call `eval.bayes_ci(R)` to get the posterior mean, posterior sigma, and credible interval.

In [ ]:
R_binary = np.array([
    [1, 1, 1, 1, 1],
    [1, 1, 1, 0, 1],
    [1, 0, 0, 1, 0],
    [0, 0, 1, 0, 0],
], dtype=int)

print("R_binary (rows=questions, columns=trials):")
print(R_binary)

mu_bayes, sigma_bayes, lo_bayes, hi_bayes = eval.bayes_ci(R_binary)

M, N = R_binary.shape
c = R_binary.sum(axis=1)
manual_row_mu = (1 + c) / (N + 2)
print("\nSuccess counts:", c)
print("Per-question means:", np.round(manual_row_mu, 4))
print(f"\nBayes@{N}: mu={mu_bayes:.4f}, sigma={sigma_bayes:.4f}, 95% CrI=[{lo_bayes:.4f}, {hi_bayes:.4f}]")


## Example 2: Greedy Run as Prior Knowledge for `top-p` Samples

In this example, we evaluate each question once with greedy decoding and then collect several stochastic `top-p` samples. The `top-p` samples stay in `R`; the greedy outcomes go in `R0`.

`R0` must have the same number of rows as `R`. Each column is a prior observation for the corresponding question. We compute Bayes@$N$ from `top-p` alone, then recompute it with the greedy run included as prior evidence.


In [ ]:
N = 7
top_p_runs = np.array([
    [1, 1, 1, 1, 0, 1, 1],
    [1, 0, 0, 1, 0, 0, 1],
    [0, 0, 0, 0, 1, 0, 0],
    [1, 1, 1, 0, 1, 1, 0],
    [0, 0, 1, 0, 0, 0, 0],
], dtype=int)

greedy_prior = np.array([
    [1],
    [1],
    [0],
    [1],
    [0],
], dtype=int)

ci_top_p_only = eval.bayes_ci(top_p_runs)
ci_with_prior = eval.bayes_ci(top_p_runs, R0=greedy_prior)

c_top_p = top_p_runs.sum(axis=1)
g = greedy_prior[:, 0]
row_mu_top_p_only = (1 + c_top_p) / (N + 2)
row_mu_with_prior = (1 + g + c_top_p) / (N + greedy_prior.shape[1] + 2)

print("Top-p result matrix R (rows=questions, columns=7 sampled trials):")
print(top_p_runs)
print("\nGreedy prior matrix R0 (one prior observation per question):")
print(greedy_prior)
print("\nTop-p success counts:", c_top_p)
print("Greedy outcomes:", g)
print("Means without R0:", np.round(row_mu_top_p_only, 4))
print("Means with R0:", np.round(row_mu_with_prior, 4))
print()
print_ci("Bayes@7 from top-p only", ci_top_p_only)
print_ci("Bayes@7 with greedy prior", ci_with_prior)


## Example 3: Categorical Evaluation with Correctness and `token_ratio`

In this example, each attempt is mapped to a rubric category. We pass the category matrix as `R_cat` and the category weights as `w`.

`correctness` says whether the answer is right. `token_ratio` is used as a rough efficiency signal. Correct and efficient answers get the highest category; correct but verbose answers still get credit, but less. The category weights are

$$
w = (0.0, 0.2, 0.75, 1.0).
$$

Bayes@$N$ averages the posterior category probabilities with those weights:

$$
\bar{\pi} = \frac{1}{M} \sum_{\alpha=1}^{M} \sum_{k=0}^{C} w_k \pi_{\alpha k}.
$$

We compare a correctness-only score against the rubric-aware score.

- `0`: wrong and verbose
- `1`: wrong and efficient
- `2`: correct and verbose
- `3`: correct and efficient


In [ ]:
correctness = np.array([
    [1, 1, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 1],
    [1, 1, 0, 0],
], dtype=int)

token_ratio = np.array([
    [0.80, 1.60, 0.90, 1.10],
    [1.40, 1.00, 1.70, 0.70],
    [1.80, 0.90, 1.40, 0.95],
    [0.85, 1.30, 1.10, 1.60],
])

length_threshold = 1.20
verbose = token_ratio > length_threshold

# 0 = wrong & verbose, 1 = wrong & efficient,
# 2 = correct & verbose, 3 = correct & efficient
R_cat = np.where(correctness == 1, np.where(verbose, 2, 3), np.where(verbose, 0, 1)).astype(int)
w = np.array([0.0, 0.2, 0.75, 1.0])

mu_binary, sigma_binary = eval.bayes(correctness)
mu_cat, sigma_cat = eval.bayes(R_cat, w)
ci_cat = eval.bayes_ci(R_cat, w)

print("Category legend:")
print("  0 = wrong & verbose")
print("  1 = wrong & efficient")
print("  2 = correct & verbose")
print("  3 = correct & efficient")
print("\nCorrectness matrix:")
print(correctness)
print("\nToken-ratio matrix:")
print(token_ratio)
print(f"\nLength threshold = {length_threshold:.2f}")
print("\nCategorical result matrix R_cat:")
print(R_cat)
print("Category counts:", np.bincount(R_cat.ravel(), minlength=4))
print(f"\nCorrectness-only Bayes@{correctness.shape[1]}: mu={mu_binary:.4f}, sigma={sigma_binary:.4f}")
print(f"Rubric-aware Bayes@{R_cat.shape[1]}: mu={mu_cat:.4f}, sigma={sigma_cat:.4f}")
print_ci("Rubric-aware Bayes@N interval", ci_cat)


## Example 4: Uncertainty with Bayes@N

In this example, we compute a credible interval for each model and then compare model pairs. Point estimates alone can make small gaps look more decisive than they are.

We first check whether intervals overlap. Then we compute a ranking-confidence value using the normal approximation from the helper function above:

$$
z = \frac{|\mu - \mu'|}{\sqrt{\sigma^2 + (\sigma')^2}}, \qquad
\rho = \frac{1}{2}\left(1 + \operatorname{erf}\left(\frac{z}{\sqrt{2}}\right)\right).
$$


In [ ]:
N = 6
model_A = matrix_from_counts([5, 4, 4, 3, 5, 4], n_trials=N)
model_B = matrix_from_counts([4, 4, 3, 3, 5, 4], n_trials=N)
model_C = matrix_from_counts([2, 2, 3, 2, 1, 2], n_trials=N)

ci_A = eval.bayes_ci(model_A)
ci_B = eval.bayes_ci(model_B)
ci_C = eval.bayes_ci(model_C)

print_ci("Model A", ci_A)
print_ci("Model B", ci_B)
print_ci("Model C", ci_C)

z_ab, rho_ab = ranking_confidence(ci_A[0], ci_A[1], ci_B[0], ci_B[1])
z_ac, rho_ac = ranking_confidence(ci_A[0], ci_A[1], ci_C[0], ci_C[1])

print("\nPairwise interpretation:")
print(f"A vs B -> intervals overlap: {intervals_overlap(ci_A, ci_B)}, z={z_ab:.3f}, ranking confidence={rho_ab:.3f}")
print(f"A vs C -> intervals overlap: {intervals_overlap(ci_A, ci_C)}, z={z_ac:.3f}, ranking confidence={rho_ac:.3f}")


## Example 5: Intervals for `Pass@k` and `mG-Pass@k`

In this example, we compare observed pass-family point estimates with interval-aware posterior estimates. `Pass@k` asks whether at least one of `k` sampled attempts succeeds.

`pass_at_k` and `mg_pass_at_k` return observed point estimates from the matrix. Their `_ci` variants return posterior mean, sigma, and credible interval.


In [ ]:
k = 4

lucky_model = np.array([
    [1, 0, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0, 0],
    [1, 0, 1, 0, 0, 0],
], dtype=int)

steady_model = np.array([
    [1, 1, 1, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 1, 0, 0, 0],
], dtype=int)

for name, R in [("Lucky", lucky_model), ("Steady", steady_model)]:
    print(f"{name} model")
    mu_b, sigma_b = eval.bayes(R)
    pass_point = eval.pass_at_k(R, k=k)
    pass_ci = eval.pass_at_k_ci(R, k=k)
    mg_point = eval.mg_pass_at_k(R, k=k)
    mg_ci = eval.mg_pass_at_k_ci(R, k=k)

    print(f"  Bayes@{R.shape[1]}: mu={mu_b:.4f}, sigma={sigma_b:.4f}")
    print(f"  Observed Pass@{k}: {pass_point:.4f}")
    print(f"  Posterior Pass@{k}: mu={pass_ci[0]:.4f}, sigma={pass_ci[1]:.4f}, 95% CrI=[{pass_ci[2]:.4f}, {pass_ci[3]:.4f}]")
    print(f"  Observed mG-Pass@{k}: {mg_point:.4f}")
    print(f"  Posterior mG-Pass@{k}: mu={mg_ci[0]:.4f}, sigma={mg_ci[1]:.4f}, 95% CrI=[{mg_ci[2]:.4f}, {mg_ci[3]:.4f}]")
    print()


## Takeaways

- **Bayes@N** gives a posterior mean and uncertainty for both binary and categorical evaluation.
- If you have prior outcomes such as one greedy decoding run per question, pass them through `R0` so Bayes@N can incorporate that prior knowledge.
- Under the uniform prior, **Bayes@N** and **avg@N** produce the same ranking, but Bayes@N makes uncertainty explicit.
- The categorical formulation is useful when raw LLM outputs carry more signals than just correctness.
- If you still report pass-family metrics, use their interval-aware counterparts in `scorio` so you can distinguish lucky finite-sample wins from stable behavior.
